# Litter Detection — Analysis Notebook

Use this notebook to:
1. Inspect the prepared training data (images + masks)
2. Review MLflow experiment history
3. Run inference with the best trained model on arbitrary images

**Prerequisites**: run `python prepare.py` at least once so `data/` exists.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

DATA_DIR   = Path('data')
IMAGES_DIR = DATA_DIR / 'images'
MASKS_DIR  = DATA_DIR / 'masks'

print('Data directory exists:', DATA_DIR.exists())

## 1. Dataset overview

In [ ]:
meta_path = DATA_DIR / 'meta.json'
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print(json.dumps(meta, indent=2))
else:
    print('meta.json not found — run prepare.py first')

In [ ]:
train_stems = (DATA_DIR / 'train.txt').read_text().splitlines()
val_stems   = (DATA_DIR / 'val.txt').read_text().splitlines()
print(f'Train: {len(train_stems)}   Val: {len(val_stems)}')

## 2. Sample images + masks

In [ ]:
def show_samples(stems, n=8, title='Samples'):
    chosen = random.sample(stems, min(n, len(stems)))
    fig, axes = plt.subplots(2, len(chosen), figsize=(len(chosen) * 2.5, 5))
    fig.suptitle(title, fontsize=13)
    for col, stem in enumerate(chosen):
        img  = Image.open(IMAGES_DIR / f'{stem}.jpg')
        mask = Image.open(MASKS_DIR  / f'{stem}.png')
        axes[0, col].imshow(img)
        axes[0, col].axis('off')
        axes[0, col].set_title(stem, fontsize=7)
        # overlay mask on image
        overlay = np.array(img).copy()
        m = np.array(mask) > 0
        overlay[m] = overlay[m] * 0.4 + np.array([255, 80, 0]) * 0.6
        axes[1, col].imshow(overlay.astype(np.uint8))
        axes[1, col].axis('off')
        frac = m.mean()
        axes[1, col].set_title(f'{frac:.1%} litter', fontsize=7)
    plt.tight_layout()
    plt.show()

show_samples(train_stems, n=8, title='Training samples (image / litter overlay)')

In [ ]:
show_samples(val_stems, n=8, title='Validation samples')

## 3. Mask statistics

In [ ]:
fracs = []
all_stems = train_stems + val_stems
for stem in all_stems:
    m = np.array(Image.open(MASKS_DIR / f'{stem}.png'))
    fracs.append(m.mean())

fracs = np.array(fracs)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fracs, bins=50, edgecolor='black')
axes[0].set_xlabel('Litter pixel fraction')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of litter coverage per image')

axes[1].hist(fracs, bins=50, cumulative=True, density=True, edgecolor='black')
axes[1].set_xlabel('Litter pixel fraction')
axes[1].set_ylabel('Cumulative density')
axes[1].set_title('CDF of litter coverage')

plt.tight_layout()
plt.show()

print(f'Mean:   {fracs.mean():.3%}')
print(f'Median: {np.median(fracs):.3%}')
print(f'Empty masks (0%): {(fracs == 0).sum()}')
print(f'>50% litter:      {(fracs > 0.5).sum()}')

## 4. MLflow experiment history

In [ ]:
import mlflow

client = mlflow.MlflowClient()

try:
    exp = client.get_experiment_by_name('litter-segmentation')
    if exp is None:
        print('No experiment named "litter-segmentation" found yet.')
        print('Run train.py to create one.')
    else:
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=['metrics.best_val_iou DESC']
        )
        rows = []
        for r in runs:
            rows.append({
                'run_name':     r.data.tags.get('mlflow.runName', r.info.run_id[:8]),
                'best_val_iou': r.data.metrics.get('best_val_iou', float('nan')),
                'val_iou':      r.data.metrics.get('val_iou',      float('nan')),
                'elapsed_s':    r.data.metrics.get('elapsed_s',    float('nan')),
                'batch_size':   r.data.params.get('batch_size',    '?'),
                'lr':           r.data.params.get('lr',            '?'),
                'encoder_ch':   r.data.params.get('encoder_channels', '?'),
            })
        import pandas as pd
        df = pd.DataFrame(rows)
        display(df.style.highlight_max(subset=['best_val_iou'], color='lightgreen'))
except Exception as e:
    print(f'MLflow error: {e}')

In [ ]:
# Plot val_iou over epochs for all runs
try:
    exp = client.get_experiment_by_name('litter-segmentation')
    if exp:
        runs = client.search_runs(experiment_ids=[exp.experiment_id])
        fig, ax = plt.subplots(figsize=(12, 5))
        for r in runs:
            name = r.data.tags.get('mlflow.runName', r.info.run_id[:8])
            history = client.get_metric_history(r.info.run_id, 'val_iou')
            if history:
                steps = [h.step for h in history]
                ious  = [h.value for h in history]
                ax.plot(steps, ious, label=name, marker='o', markersize=3)
        ax.set_xlabel('Step')
        ax.set_ylabel('val_iou')
        ax.set_title('Validation IoU across experiments')
        ax.legend()
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f'Could not plot history: {e}')

## 5. Inference with best model

In [ ]:
# Import the model definition from train.py
import sys
sys.path.insert(0, '.')
from train import UNet, ENCODER_CHANNELS, DECODER_CHANNELS

MODEL_PATH = Path('best_model.pth')

def get_device():
    if torch.cuda.is_available():  return torch.device('cuda')
    if torch.backends.mps.is_available(): return torch.device('mps')
    return torch.device('cpu')

device = get_device()

if MODEL_PATH.exists():
    model = UNet(encoder_channels=ENCODER_CHANNELS,
                 decoder_channels=DECODER_CHANNELS).to(device)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f'Model loaded from {MODEL_PATH} on {device}')
else:
    print(f'{MODEL_PATH} not found — run train.py first')
    model = None

In [ ]:
from albumentations import Compose, Resize, Normalize
from albumentations.pytorch import ToTensorV2

INFER_SIZE = 512

infer_transform = Compose([
    Resize(INFER_SIZE, INFER_SIZE),
    Normalize(mean=(0.485, 0.456, 0.406),
              std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

def predict(image_path: str, threshold: float = 0.5):
    """Run inference on an image file (path string or PIL Image)."""
    assert model is not None, 'Load a model first'
    if isinstance(image_path, str):
        img = np.array(Image.open(image_path).convert('RGB'))
    else:
        img = np.array(image_path.convert('RGB'))
    tensor = infer_transform(image=img)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        logit = model(tensor)
    prob = torch.sigmoid(logit).squeeze().cpu().numpy()
    mask = (prob > threshold).astype(np.uint8)
    return img, prob, mask


def show_prediction(image_path: str, threshold: float = 0.5, gt_mask_path: str = None):
    img, prob, pred_mask = predict(image_path, threshold)
    img_resized = np.array(Image.fromarray(img).resize((INFER_SIZE, INFER_SIZE)))

    ncols = 4 if gt_mask_path else 3
    fig, axes = plt.subplots(1, ncols, figsize=(ncols * 4, 4))

    axes[0].imshow(img_resized); axes[0].set_title('Input'); axes[0].axis('off')
    axes[1].imshow(prob, cmap='hot', vmin=0, vmax=1)
    axes[1].set_title('Probability map'); axes[1].axis('off')

    overlay = img_resized.copy().astype(float)
    overlay[pred_mask.astype(bool)] = (
        overlay[pred_mask.astype(bool)] * 0.4 + np.array([255, 80, 0]) * 0.6
    )
    axes[2].imshow(overlay.astype(np.uint8)); axes[2].set_title('Prediction overlay'); axes[2].axis('off')

    if gt_mask_path:
        gt = np.array(Image.open(gt_mask_path).resize((INFER_SIZE, INFER_SIZE),
                                                        Image.NEAREST))
        axes[3].imshow(gt, cmap='gray'); axes[3].set_title('Ground truth'); axes[3].axis('off')
        inter = float((pred_mask * gt).sum())
        union = float(((pred_mask + gt) > 0).sum())
        iou = inter / max(union, 1)
        fig.suptitle(f'IoU = {iou:.4f}', fontsize=12)

    plt.tight_layout()
    plt.show()

print('predict() and show_prediction() ready')

In [ ]:
# Show predictions on validation samples
if model is not None:
    sample_stems = random.sample(val_stems, min(4, len(val_stems)))
    for stem in sample_stems:
        show_prediction(
            str(IMAGES_DIR / f'{stem}.jpg'),
            gt_mask_path=str(MASKS_DIR / f'{stem}.png')
        )

In [ ]:
# ── Inference on an arbitrary image ──────────────────────────────────────────
# Change the path below to any image you want to test

# CUSTOM_IMAGE = '/path/to/your/image.jpg'
# show_prediction(CUSTOM_IMAGE)

## 6. Batch validation IoU

In [ ]:
if model is not None:
    ious = []
    for stem in val_stems:
        img_path  = str(IMAGES_DIR / f'{stem}.jpg')
        mask_path = str(MASKS_DIR  / f'{stem}.png')
        _, prob, pred = predict(img_path)
        gt = (np.array(Image.open(mask_path).resize(
            (INFER_SIZE, INFER_SIZE), Image.NEAREST)) > 0).astype(np.uint8)
        inter = float((pred * gt).sum())
        union = float(((pred + gt) > 0).sum())
        ious.append(inter / max(union, 1))

    ious = np.array(ious)
    print(f'Val IoU — mean: {ious.mean():.4f}  median: {np.median(ious):.4f}  '
          f'min: {ious.min():.4f}  max: {ious.max():.4f}')

    plt.figure(figsize=(8, 4))
    plt.hist(ious, bins=30, edgecolor='black')
    plt.xlabel('IoU per image')
    plt.ylabel('Count')
    plt.title('Distribution of per-image IoU on validation set')
    plt.axvline(ious.mean(), color='red', linestyle='--', label=f'mean={ious.mean():.3f}')
    plt.legend()
    plt.tight_layout()
    plt.show()